# Bike Sharing Demand — Data Understanding

This notebook performs the initial understanding and quality assessment of the UCI Bike Sharing Dataset.

The main objectives are:

- Load the raw hourly bike-sharing data.
- Understand the dataset structure and variables.
- Inspect data types and missing values.
- Check for duplicate records.
- Examine descriptive statistics.
- Understand the target variable (`cnt`).
- Identify potential data-quality issues.
- Identify variables that should not be used as predictors because of target leakage.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

## 1. Load the Dataset

The analysis uses the hourly dataset (`hour.csv`) provided by the UCI Machine Learning Repository.

The dataset contains hourly bike rental records from the Capital Bikeshare system in Washington, D.C., covering 2011 and 2012.

In [3]:
DATA_PATH = "../data/raw/hour.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.29,0.81,0.00,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.27,0.80,0.00,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.27,0.80,0.00,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.29,0.75,0.00,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.29,0.75,0.00,0,1,1


In [4]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 17,379
Columns: 17


## 2. Dataset Overview

First, we inspect the overall structure of the dataset and identify the available variables.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


In [6]:
df.columns.tolist()

['instant',
 'dteday',
 'season',
 'yr',
 'mnth',
 'hr',
 'holiday',
 'weekday',
 'workingday',
 'weathersit',
 'temp',
 'atemp',
 'hum',
 'windspeed',
 'casual',
 'registered',
 'cnt']

### Variable Description

The main variables can be grouped into several categories:

- **Time:** `dteday`, `yr`, `mnth`, `hr`, `weekday`
- **Calendar:** `season`, `holiday`, `workingday`
- **Weather:** `weathersit`, `temp`, `atemp`, `hum`, `windspeed`
- **User type:** `casual`, `registered`
- **Target:** `cnt`
- **Identifier:** `instant`

## 3. Data Types

We inspect the data types of all variables to determine whether any variables require conversion or special handling.

In [7]:
dtype_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_values": df.isna().sum().values,
    "unique_values": df.nunique().values
})

dtype_summary

,column,dtype,missing_values,unique_values
0,instant,int64,0,17379
1,dteday,object,0,731
2,season,int64,0,4
3,yr,int64,0,2
4,mnth,int64,0,12
5,hr,int64,0,24
6,holiday,int64,0,2
7,weekday,int64,0,7
8,workingday,int64,0,2
9,weathersit,int64,0,4


## 4. Missing Values

Missing values can affect both exploratory analysis and predictive modeling.

We check every column for missing observations.

In [8]:
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_values"] / len(df) * 100
)

missing_summary

,missing_values,missing_percentage
instant,0,0.00
weathersit,0,0.00
registered,0,0.00
casual,0,0.00
windspeed,0,0.00
hum,0,0.00
atemp,0,0.00
temp,0,0.00
workingday,0,0.00
dteday,0,0.00


In [9]:
total_missing = df.isna().sum().sum()

print(f"Total missing values: {total_missing}")

Total missing values: 0


## 5. Duplicate Records

We check whether the dataset contains duplicated rows.

In [10]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


## 6. Descriptive Statistics

Descriptive statistics provide an initial understanding of the numerical variables, including their central tendency and dispersion.

In [11]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
instant,17379.00,8690.00,5017.03,1.00,4345.50,8690.00,13034.50,17379.00
season,17379.00,2.50,1.11,1.00,2.00,3.00,3.00,4.00
yr,17379.00,0.50,0.50,0.00,0.00,1.00,1.00,1.00
mnth,17379.00,6.54,3.44,1.00,4.00,7.00,10.00,12.00
hr,17379.00,11.55,6.91,0.00,6.00,12.00,18.00,23.00
holiday,17379.00,0.03,0.17,0.00,0.00,0.00,0.00,1.00
weekday,17379.00,3.00,2.01,0.00,1.00,3.00,5.00,6.00
workingday,17379.00,0.68,0.47,0.00,0.00,1.00,1.00,1.00
weathersit,17379.00,1.43,0.64,1.00,1.00,1.00,2.00,4.00
temp,17379.00,0.50,0.19,0.02,0.34,0.50,0.66,1.00


In [12]:
df.describe(include="object").T

,count,unique,top,freq
dteday,17379,731,2011-01-01,24


## 7. Target Variable: `cnt`

The target variable `cnt` represents the total number of bike rentals during each hour.

It is defined as:

`cnt = casual + registered`

Therefore, `casual` and `registered` will not be used as predictors when modeling `cnt`, because doing so would introduce target leakage.

In [13]:
target_summary = df["cnt"].describe()

target_summary

count   17379.00
mean      189.46
std       181.39
min         1.00
25%        40.00
50%       142.00
75%       281.00
max       977.00
Name: cnt, dtype: float64

In [14]:
print("Minimum hourly rentals:", df["cnt"].min())
print("Maximum hourly rentals:", df["cnt"].max())
print("Mean hourly rentals:", df["cnt"].mean())
print("Median hourly rentals:", df["cnt"].median())

Minimum hourly rentals: 1
Maximum hourly rentals: 977
Mean hourly rentals: 189.46308763450142
Median hourly rentals: 142.0


In [15]:
leakage_check = np.allclose(
    df["cnt"],
    df["casual"] + df["registered"]
)

print(f"cnt = casual + registered: {leakage_check}")

cnt = casual + registered: True


## 8. Identifier Check

The `instant` column is a record index rather than a meaningful explanatory variable.

It will therefore not be used as a predictor in the machine-learning models.

In [16]:
print("Number of unique instant values:", df["instant"].nunique())
print("Number of rows:", len(df))
print("Is instant unique:", df["instant"].is_unique)

Number of unique instant values: 17379
Number of rows: 17379
Is instant unique: True


## 9. Categorical Variables

Several variables represent categories rather than continuous numerical measurements.

We inspect their unique values and frequencies.

In [17]:
categorical_columns = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit"
]

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts().sort_index())


season
season
1    4242
2    4409
3    4496
4    4232
Name: count, dtype: int64

yr
yr
0    8645
1    8734
Name: count, dtype: int64

mnth
mnth
1     1429
2     1341
3     1473
4     1437
5     1488
6     1440
7     1488
8     1475
9     1437
10    1451
11    1437
12    1483
Name: count, dtype: int64

hr
hr
0     726
1     724
2     715
3     697
4     697
5     717
6     725
7     727
8     727
9     727
10    727
11    727
12    728
13    729
14    729
15    729
16    730
17    730
18    728
19    728
20    728
21    728
22    728
23    728
Name: count, dtype: int64

holiday
holiday
0    16879
1      500
Name: count, dtype: int64

weekday
weekday
0    2502
1    2479
2    2453
3    2475
4    2471
5    2487
6    2512
Name: count, dtype: int64

workingday
workingday
0     5514
1    11865
Name: count, dtype: int64

weathersit
weathersit
1    11413
2     4544
3     1419
4        3
Name: count, dtype: int64


## 10. Weather Variables

The dataset contains normalized weather-related variables:

- `temp`: normalized temperature
- `atemp`: normalized feeling temperature
- `hum`: normalized humidity
- `windspeed`: normalized wind speed

The `weathersit` variable represents weather conditions using categorical levels.

In [18]:
weather_columns = [
    "temp",
    "atemp",
    "hum",
    "windspeed"
]

df[weather_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
temp,17379.00,0.50,0.19,0.02,0.34,0.50,0.66,1.00
atemp,17379.00,0.48,0.17,0.00,0.33,0.48,0.62,1.00
hum,17379.00,0.63,0.19,0.00,0.48,0.63,0.78,1.00
windspeed,17379.00,0.19,0.12,0.00,0.10,0.19,0.25,0.85


## 11. Date Range

The `dteday` column represents the date associated with each hourly observation.

We convert it to a datetime representation and inspect the temporal coverage of the dataset.

In [19]:
df["dteday"] = pd.to_datetime(df["dteday"])

print("Start date:", df["dteday"].min().date())
print("End date:", df["dteday"].max().date())

Start date: 2011-01-01
End date: 2012-12-31


## 12. Initial Data Quality Assessment

Based on the initial inspection, we evaluate:

- Dataset dimensions
- Missing values
- Duplicate records
- Data types
- Identifier variables
- Target variable definition
- Potential target leakage
- Categorical variables
- Weather-related variables
- Temporal coverage

These checks establish the foundation for the exploratory analysis performed in the next notebook.

## 13. Conclusions

The dataset provides hourly observations of bike-sharing demand together with temporal, calendar, and weather-related variables.

The target variable is `cnt`, representing total hourly rentals.

Two important modeling considerations have been identified:

1. `instant` is an identifier and should not be used as a predictive feature.
2. `casual` and `registered` directly determine `cnt` and therefore must be excluded from the predictive feature set to avoid target leakage.

The dataset is now ready for exploratory data analysis.